# 02 — Fairness Methods

scikit-fair provides three types of fairness methods. They differ in how they
plug into a modelling pipeline, so this notebook introduces them from the most
familiar pattern to the least:

1. **Transformers** — standard sklearn `fit` / `transform`, drop into any `sklearn.pipeline.Pipeline`
2. **Samplers** — change the number of rows, so they need an `imblearn.Pipeline` instead
3. **Meta-estimators** — wrap the classifier itself, so they replace the last pipeline step

In [1]:
from skfair.datasets import load_ricci
from skfair.metrics import accuracy, statistical_parity_difference
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = load_ricci(preprocessed=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## Transformer — LearningFairRepresentations

Transformers implement the standard sklearn `fit` / `transform` interface.
They learn a fair representation of the features, then you train any classifier
on the transformed data. Because they don't change the number of rows, they
work inside a regular `sklearn.pipeline.Pipeline`.

In [2]:
from skfair.preprocessing import LearningFairRepresentations

lfr = LearningFairRepresentations(sens_attr="Race", priv_group=1, random_state=42)
lfr.fit(X_train, y_train)
X_train_fair = lfr.transform(X_train)
X_test_fair = lfr.transform(X_test)
print("Transformed shape:", X_train_fair.shape)

Transformed shape: (82, 4)


Since LFR is a regular transformer, it fits into a standard sklearn pipeline:

In [3]:
from sklearn.pipeline import Pipeline

pipe_lfr = Pipeline([
    ("lfr", LearningFairRepresentations(sens_attr="Race", priv_group=1, random_state=42)),
    ("clf", LogisticRegression(solver="liblinear", max_iter=1000, random_state=42)),
])
pipe_lfr.fit(X_train, y_train)
y_pred_lfr = pipe_lfr.predict(X_test)

print("Accuracy:", round(accuracy(y_test, y_pred_lfr), 3))
print("SPD:", round(statistical_parity_difference(y_test, y_pred_lfr, X_test["Race"]), 3))

Accuracy: 0.944
SPD: -0.371


## Sampler — FairSmote

Samplers balance the dataset by generating or removing rows via
`fit_resample(X, y)`. Because they change the number of samples, a regular
`sklearn.pipeline.Pipeline` will reject them. The fix is simple: use
`imblearn.pipeline.Pipeline` instead — the API is identical, it just knows
how to call `fit_resample` during `fit`.

In [4]:
from skfair.preprocessing import FairSmote

fs = FairSmote(sens_attr="Race", random_state=42)
X_res, y_res = fs.fit_resample(X_train, y_train)
print("Before:", X_train.shape, "After:", X_res.shape)

Before: (82, 4) After: (116, 4)


Using FairSmote inside an imblearn pipeline — same API as sklearn, just swap
the import:

In [5]:
from imblearn.pipeline import Pipeline as ImbPipeline  # only difference

pipe_fs = ImbPipeline([
    ("sampler", FairSmote(sens_attr="Race", random_state=42)),
    ("clf", LogisticRegression(solver="liblinear", max_iter=1000, random_state=42)),
])
pipe_fs.fit(X_train, y_train)
y_pred_fs = pipe_fs.predict(X_test)

print("Accuracy:", round(accuracy(y_test, y_pred_fs), 3))
print("SPD:", round(statistical_parity_difference(y_test, y_pred_fs, X_test["Race"]), 3))

Accuracy: 0.75
SPD: -0.267


## Meta-estimator — ReweighingClassifier

Meta-estimators go one step further: they absorb the classifier. A
`ReweighingClassifier` wraps a sklearn estimator, computes fairness-aware
sample weights internally, and exposes `fit` / `predict` directly.

Because the fairness logic and the classifier are fused into one object, they
don't fit as a preprocessing step in either pipeline type. Instead, you use
them as the final (or only) estimator. If you need upstream preprocessing
you can still place a meta-estimator at the end of a pipeline.

In [6]:
from skfair.preprocessing import ReweighingClassifier

rw = ReweighingClassifier(
    estimator=LogisticRegression(solver="liblinear", max_iter=1000, random_state=42),
    sens_attr="Race",
)
rw.fit(X_train, y_train)
y_pred_rw = rw.predict(X_test)

print("Accuracy:", round(accuracy(y_test, y_pred_rw), 3))
print("SPD:", round(statistical_parity_difference(y_test, y_pred_rw, X_test["Race"]), 3))

Accuracy: 0.75
SPD: -0.267


## Summary

| Pattern | Example | API | Pipeline |
|---|---|---|---|
| Transformer | LearningFairRepresentations | `fit` / `transform` | `sklearn.pipeline.Pipeline` |
| Sampler | FairSmote | `fit_resample` | `imblearn.pipeline.Pipeline` |
| Meta-estimator | ReweighingClassifier | `fit` / `predict` (wraps clf) | Place as final step in any pipeline |

## Pipeline recommendation

> **Tip:** We recommend always using `imblearn.pipeline.Pipeline` — it extends sklearn's Pipeline with `fit_resample` support, so it works with all scikit-fair methods (transformers, samplers, and meta-estimators) without needing to switch imports.